# Credora Finance — Dashboard Data Preparation

## Objective

The purpose of this notebook is to prepare a dashboard-ready dataset for the Credora Finance Credit Risk & Loan Approval Analytics Platform.

The trained credit-risk model was evaluated on a held-out test set, but the business dashboard needs to represent the complete application portfolio.

Therefore, this notebook will:

- Load the complete 7,500 loan application dataset.
- Load the trained credit-risk model pipeline.
- Apply the same model features and preprocessing used during model development.
- Generate default probabilities for all applications.
- Convert probabilities into 0–100 risk scores.
- Assign Low, Medium, and High risk categories.
- Preserve actual loan outcomes separately from model predictions.
- Create a final dataset for dashboard development.

> **Important:** Full-portfolio predictions are used for business monitoring and visualization. Model performance metrics such as ROC-AUC, recall, precision, and F1-score remain based on the held-out test set and are not recalculated using these portfolio predictions.

## 1. Load Dataset and Trained Model

The complete master dataset contains all 7,500 loan applications, including applications with both known and unknown default outcomes.

The saved Scikit-learn pipeline contains the preprocessing steps and the final trained classification model.

In [3]:
import pandas as pd
import joblib

# Load the complete 7,500-application master dataset
master_df = pd.read_csv("credora_master_7500.csv")

# Load the trained credit-risk model pipeline
final_model = joblib.load("credora_final_model.pkl")

# Basic verification
print("Master dataset shape:", master_df.shape)
print("Model loaded successfully:", type(final_model))

# Check target distribution
print("\nLoan outcome distribution:")
print(master_df["loan_default"].value_counts(dropna=False))

# Count known and unknown outcomes
known_outcomes = master_df["loan_default"].notna().sum()
unknown_outcomes = master_df["loan_default"].isna().sum()

# Display summary
print("\nTotal applications:", len(master_df))
print("Known outcomes:", known_outcomes)
print("Unknown outcomes:", unknown_outcomes)

Master dataset shape: (7500, 58)
Model loaded successfully: <class 'sklearn.pipeline.Pipeline'>

Loan outcome distribution:
loan_default
NaN    4097
0.0    2606
1.0     797
Name: count, dtype: int64

Total applications: 7500
Known outcomes: 3403
Unknown outcomes: 4097


### Key Findings

- The trained model expects **41 input features**.
- All **41 required features** are available in the full 7,500-row master dataset.
- No required model features are missing.
- The saved pipeline can therefore be applied directly to the complete portfolio using the same preprocessing structure used during model development.

## 2. Validate Model Features

Before generating portfolio risk scores, the features available in the master dataset must be checked against the features expected by the trained model.

This prevents feature mismatches and ensures that the same input structure used during model development is applied during portfolio scoring.

In [2]:
# Features used during model training
model_features = list(final_model.feature_names_in_)

print("Number of features expected by model:", len(model_features))
print("\nModel features:")
print(model_features)

# Check whether any required features are missing from master_df
missing_features = [
    col for col in model_features
    if col not in master_df.columns
]

print("\nMissing features:", missing_features)

Number of features expected by model: 41

Model features:
['loan_type', 'loan_purpose', 'requested_amount', 'loan_term_months', 'interest_rate', 'applicant_monthly_income', 'existing_emi', 'debt_to_income_ratio', 'collateral_flag', 'annual_income', 'loan_to_income', 'gender', 'age', 'marital_status', 'dependents', 'education', 'employment_type', 'occupation', 'city', 'state', 'residence_type', 'years_at_residence', 'kyc_status', 'credit_score', 'num_open_accounts', 'num_credit_inquiries_6m', 'num_late_payments_30d', 'num_late_payments_90d', 'num_defaults_prior', 'bankruptcies', 'total_credit_limit', 'total_outstanding_debt', 'oldest_account_age_months', 'debt_to_limit', 'total_inflow_6m', 'total_outflow_6m', 'emi_outflow_6m', 'avg_balance', 'transaction_count', 'net_cash_flow_6m', 'inflow_outflow_ratio']

Missing features: []


### Key Findings

- The trained model expects **41 input features**.
- All 41 required features are available in the full master dataset.
- No model features are missing.
- The saved pipeline can therefore be applied directly to all 7,500 applications using the same preprocessing structure used during model development.

## 3. Generate Portfolio Risk Predictions

The trained model is now applied to all 7,500 loan applications to estimate the probability of default.

These predictions are intended for portfolio monitoring and dashboard visualization.

The predicted probability represents the model-estimated likelihood of default. It does not replace the actual `loan_default` outcome.

Applications with unknown actual outcomes remain marked as unknown even after receiving a model-generated risk prediction.

In [4]:
# Select exactly the 41 features expected by the trained pipeline
X_portfolio = master_df[model_features].copy()

# Generate probability of the positive class (default = 1)
default_probability = final_model.predict_proba(X_portfolio)[:, 1]

print("Applications scored:", len(default_probability))
print("Minimum probability:", round(default_probability.min(), 4))
print("Maximum probability:", round(default_probability.max(), 4))
print("Average probability:", round(default_probability.mean(), 4))

Applications scored: 7500
Minimum probability: 0.0152
Maximum probability: 0.9895
Average probability: 0.6053


### Key Findings

- The trained pipeline successfully generated default probabilities for all **7,500 applications**.
- Predicted default probabilities range from **0.0152 (1.52%)** to **0.9895 (98.95%)**.
- The average predicted default probability across the full application portfolio is **0.6053 (60.53%)**.
- These values represent **model-generated risk estimates**, not confirmed default outcomes.
- Actual loan outcomes remain separately identified as known or unknown.

## 4. Create Risk Scores and Risk Categories

Predicted default probabilities are converted into an interpretable **0–100 risk score**.

Risk categories are assigned using the project-defined thresholds:

- **Low Risk:** 0–33
- **Medium Risk:** 34–66
- **High Risk:** 67–100

A higher score represents a higher model-estimated probability of default.

In [5]:
# Create dashboard working dataset
dashboard_df = master_df.copy()

# Add predicted probability
dashboard_df["default_probability"] = default_probability

# Convert probability to 0–100 risk score
dashboard_df["risk_score"] = (
    dashboard_df["default_probability"] * 100
).round().astype(int)

# Assign risk category
dashboard_df["risk_category"] = pd.cut(
    dashboard_df["risk_score"],
    bins=[-1, 33, 66, 100],
    labels=["Low", "Medium", "High"]
)

# Mark whether the actual default outcome is available
dashboard_df["outcome_status"] = dashboard_df["loan_default"].apply(
    lambda x: "Unknown" if pd.isna(x) else "Known"
)

print("Dashboard dataset shape:", dashboard_df.shape)

print("\nRisk category distribution:")
print(dashboard_df["risk_category"].value_counts().sort_index())

print("\nOutcome status:")
print(dashboard_df["outcome_status"].value_counts())

print("\nRisk score summary:")
print(dashboard_df["risk_score"].describe().round(2))

Dashboard dataset shape: (7500, 62)

Risk category distribution:
risk_category
Low       1236
Medium    2843
High      3421
Name: count, dtype: int64

Outcome status:
outcome_status
Unknown    4097
Known      3403
Name: count, dtype: int64

Risk score summary:
count    7500.00
mean       60.53
std        23.76
min         2.00
25%        41.00
50%        63.00
75%        82.00
max        99.00
Name: risk_score, dtype: float64


### Key Findings

- Risk scores were successfully generated for all **7,500 applications**.
- Risk scores range from **2 to 99**, with an average score of **60.53**.
- The median risk score is **63**.
- **1,236 applications (16.48%)** are classified as Low Risk.
- **2,843 applications (37.91%)** are classified as Medium Risk.
- **3,421 applications (45.61%)** are classified as High Risk.
- Actual outcomes remain available for **3,403 applications**, while **4,097 applications** have unknown outcomes.
- Risk categories represent model-generated estimates and should not be interpreted as confirmed default outcomes.

## 5. Validate Risk Categories Against Known Outcomes

For applications with known outcomes, the observed default rate is compared across the model-generated risk categories.

A useful risk segmentation should generally show higher observed default rates as the predicted risk category increases.

Only applications with known `loan_default` values are used for this validation.

In [6]:
# Use only applications with known actual outcomes
known_df = dashboard_df[
    dashboard_df["loan_default"].notna()
].copy()

# Calculate actual default behavior by risk category
risk_validation = (
    known_df
    .groupby("risk_category", observed=True)
    .agg(
        total_applications=("loan_default", "count"),
        actual_defaults=("loan_default", "sum"),
        average_risk_score=("risk_score", "mean")
    )
)

risk_validation["actual_default_rate_pct"] = (
    risk_validation["actual_defaults"]
    / risk_validation["total_applications"]
    * 100
).round(2)

risk_validation["average_risk_score"] = (
    risk_validation["average_risk_score"].round(2)
)

print(risk_validation)

               total_applications  actual_defaults  average_risk_score  \
risk_category                                                            
Low                          1060             72.0               23.17   
Medium                       1799            443.0               48.26   
High                          544            282.0               76.23   

               actual_default_rate_pct  
risk_category                           
Low                               6.79  
Medium                           24.62  
High                             51.84  


### Key Findings

- The model-generated risk categories show a clear increase in observed default rates across the known-outcome population.
- **Low Risk:** 1,060 applications with an observed default rate of **6.79%**.
- **Medium Risk:** 1,799 applications with an observed default rate of **24.62%**.
- **High Risk:** 544 applications with an observed default rate of **51.84%**.
- Average risk scores also increase consistently from **23.17** (Low) to **48.26** (Medium) and **76.23** (High).
- This monotonic pattern indicates that the risk segmentation is directionally meaningful for portfolio monitoring.
- This analysis includes all labeled applications, including training observations, and is therefore treated as a **dashboard sanity check rather than independent model-performance evaluation**.

## 6. Risk Distribution by Outcome Availability

The portfolio contains both applications with known historical outcomes and applications whose default outcomes are unknown.

Risk-category distributions are compared between these groups to understand how model-generated risk is distributed across the portfolio and to avoid confusing predicted risk with confirmed default behavior.

In [7]:
risk_by_outcome = pd.crosstab(
    dashboard_df["outcome_status"],
    dashboard_df["risk_category"]
)

risk_by_outcome["Total"] = risk_by_outcome.sum(axis=1)

print(risk_by_outcome)

print("\nPercentage distribution within each outcome group:")

risk_by_outcome_pct = (
    pd.crosstab(
        dashboard_df["outcome_status"],
        dashboard_df["risk_category"],
        normalize="index"
    ) * 100
).round(2)

print(risk_by_outcome_pct)

risk_category    Low  Medium  High  Total
outcome_status                           
Known           1060    1799   544   3403
Unknown          176    1044  2877   4097

Percentage distribution within each outcome group:
risk_category     Low  Medium   High
outcome_status                      
Known           31.15   52.87  15.99
Unknown          4.30   25.48  70.22


### Key Findings

- The risk distribution differs substantially between applications with known and unknown outcomes.
- Among the **3,403 known-outcome applications**, **31.15%** are Low Risk, **52.87%** are Medium Risk, and **15.99%** are High Risk.
- Among the **4,097 unknown-outcome applications**, only **4.30%** are Low Risk, while **25.48%** are Medium Risk and **70.22%** are High Risk.
- This indicates that the unknown-outcome population has a substantially different predicted-risk profile from the known-outcome population.
- Because the model was developed using applications with known outcomes, predictions for the unknown-outcome population should be interpreted as model-generated risk estimates rather than validated outcomes.

## 7. Relationship Between Outcome Availability and Approval Status

The strong difference in predicted risk between known- and unknown-outcome applications may be related to the loan decision process.

This section compares approval status with outcome availability to determine which application groups contain observed default outcomes.

In [8]:
approval_outcome_check = pd.crosstab(
    dashboard_df["approval_status"],
    dashboard_df["outcome_status"],
    margins=True
)

print(approval_outcome_check)

outcome_status   Known  Unknown   All
approval_status                      
Approved          3403        0  3403
Rejected             0     3924  3924
Under Review         0      173   173
All               3403     4097  7500


### Key Findings

- All **3,403 approved applications** have known default outcomes.
- All **3,924 rejected applications** have unknown default outcomes.
- All **173 applications under review** also have unknown default outcomes.
- Therefore, the **4,097 unknown outcomes** correspond exactly to rejected and under-review applications.
- Historical default-rate analysis should only use approved applications with known outcomes.
- Model-generated risk scores can be displayed for all 7,500 applications, but risk predictions for rejected and under-review applications do not have observed default outcomes available for validation.
- The dashboard will therefore clearly distinguish **actual historical outcomes** from **model-predicted portfolio risk**.

## 8. Validate Dashboard Fields

Before exporting the final dashboard dataset, the required business, geographic, temporal, credit-risk, and model-generated fields are verified.

These fields will support dashboard KPIs, filters, trend analysis, geographic analysis, risk segmentation, and the high-risk application watchlist.

In [9]:
dashboard_fields = [
    "application_id",
    "customer_id",
    "application_date",
    "loan_type",
    "loan_purpose",
    "requested_amount",
    "approval_status",
    "state",
    "credit_score",
    "debt_to_income_ratio",
    "loan_default",
    "default_probability",
    "risk_score",
    "risk_category",
    "outcome_status"
]

available_fields = [
    col for col in dashboard_fields
    if col in dashboard_df.columns
]

missing_dashboard_fields = [
    col for col in dashboard_fields
    if col not in dashboard_df.columns
]

print("Required dashboard fields:", len(dashboard_fields))
print("Available fields:", len(available_fields))
print("Missing fields:", missing_dashboard_fields)

Required dashboard fields: 15
Available fields: 15
Missing fields: []


### Key Findings

- All **15 required dashboard fields** are available.
- No required dashboard fields are missing.
- The dataset contains the identifiers, application details, geographic attributes, credit-risk variables, actual outcomes, and model-generated risk outputs needed for dashboard development.
- The data is ready for final dashboard export.

## 9. Export Dashboard-Ready Dataset

The final dashboard dataset combines business application information with model-generated risk predictions.

The dataset preserves actual loan outcomes separately from predicted risk so that historical performance and model-based portfolio monitoring can be presented without confusion.

This dataset will serve as the primary data source for the Credora Finance interactive dashboard.

In [10]:
# Save final dashboard-ready dataset
dashboard_df.to_csv(
    "credora_dashboard_dataset.csv",
    index=False
)

print("Dashboard dataset saved successfully.")
print("Final shape:", dashboard_df.shape)

print("\nMissing values in risk outputs:")
print(
    dashboard_df[
        ["default_probability", "risk_score", "risk_category"]
    ].isna().sum()
)

print("\nDuplicate application IDs:")
print(dashboard_df["application_id"].duplicated().sum())

Dashboard dataset saved successfully.
Final shape: (7500, 62)

Missing values in risk outputs:
default_probability    0
risk_score             0
risk_category          0
dtype: int64

Duplicate application IDs:
0


In [11]:
from google.colab import files

files.download("credora_dashboard_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Dashboard Data Preparation Summary

The dashboard data preparation stage successfully extended the trained credit-risk model to the complete Credora Finance application portfolio.

### Final Dataset

- **Total applications:** 7,500
- **Final dashboard columns:** 62
- **Known loan outcomes:** 3,403
- **Unknown loan outcomes:** 4,097
- **Duplicate application IDs:** 0
- **Missing model risk outputs:** 0

### Portfolio Risk Distribution

- **Low Risk:** 1,236 applications (16.48%)
- **Medium Risk:** 2,843 applications (37.91%)
- **High Risk:** 3,421 applications (45.61%)

### Risk Validation

Among applications with known outcomes, observed default rates increased consistently across the model-generated risk categories:

- **Low Risk:** 6.79%
- **Medium Risk:** 24.62%
- **High Risk:** 51.84%

This provides a useful sanity check that higher model-generated risk categories correspond to higher observed default rates within the labeled population.

### Outcome Availability

All **3,403 approved applications** have known default outcomes, while all **3,924 rejected** and **173 under-review applications** have unknown outcomes.

Therefore, the dashboard will distinguish between:

- **Historical business outcomes**, calculated only where actual outcomes are available.
- **Model-generated risk estimates**, available for all 7,500 applications.

Predicted risk for rejected and under-review applications should not be interpreted as a confirmed default outcome because their actual default behavior is not observed.

### Final Output

The final dashboard-ready dataset is saved as:

`credora_dashboard_dataset.csv`

This dataset can now be used to build the Credora Finance interactive credit-risk and loan-approval dashboard.